# Demo on Colab (A100)

Colab variant of `run_demo.ipynb`. Use this to prove the brain works on a clean GPU, isolated from any local install issues. If models work here but not on your local machine, the bug is in your local CUDA / bitsandbytes setup. If they don't work here either, the bug is in the wrapper code.

Key differences from the local version:
- Clones the repo into the Colab runtime (Colab starts empty each session).
- Loads Qwen3-VL at fp16 (~18 GB) instead of 4-bit. A100 has 40 GB so this fits with headroom and skips the bitsandbytes risk entirely.
- Webcam capture replaced with `files.upload()` — Colab is a remote container, no camera access. Pre-take a handful of phone photos of the scene and upload them one at a time.

**Runtime → Change runtime type → A100 GPU** before running.

## Setup the Colab runtime (run once per session)

In [ ]:
!git clone https://github.com/Ndgandhi23/RU-IEEE-Hackathon.git
%cd RU-IEEE-Hackathon
!pip install -q -U git+https://github.com/huggingface/transformers@main accelerate bitsandbytes pillow

## Load the brain (run once)

Loads OWLv2 (~300 MB) + Qwen3-VL-8B at fp16 (~18 GB) + the controller. Cold start is 2-5 min. After this, every step cell run is fast.

In [ ]:
%matplotlib inline
import sys, time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Rectangle

REPO = Path.cwd()
sys.path.insert(0, str(REPO))

assert torch.cuda.is_available(), 'no GPU — Runtime → Change runtime type → A100'
print(f'GPU: {torch.cuda.get_device_name(0)}  ·  '
      f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM')

from brain.perception.target_finder import TargetFinder
from brain.perception.vlm_scout import VLMScout
from brain.control.loop import Action, ApproachController
from brain.control.action_to_pwm import pwm_for

REF = REPO / 'references' / 'ref.jpg'
CTX = REPO / 'references' / 'ctx.jpg'
assert REF.exists() and CTX.exists(), 'reference images missing — they should be in the cloned repo'

print('loading OWLv2...')
finder = TargetFinder(model_name='google/owlv2-base-patch16-ensemble', min_sim=0.3)
finder.load_reference(cv2.imread(str(REF)))

print('loading Qwen3-VL-8B at fp16 (no quantization — A100 has the headroom)...')
t0 = time.monotonic()
scout = VLMScout(load_in_4bit=False)
print(f'  loaded in {time.monotonic() - t0:.1f}s  ·  '
      f'VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

controller = ApproachController(
    target_finder=finder,
    vlm_scout=scout,
    reference_photo=str(REF),
    reporter_photo=str(CTX),
)
history = []
print('\nready. Run the step cell below to upload a frame and see the Action.')

## Tunables

Adjust before re-running the step cell.

In [ ]:
MIN_SIM = 0.3   # raise if OWLv2 saturates with false positives
finder.min_sim = MIN_SIM

## Step (upload a frame, see what the brain decides)

Pre-take a handful of phone photos of the bottle scene at different framings (bottle close / far / hidden / off-center). Upload one at a time when prompted. Each run shows you the chosen Action with the same color coding as the local demo.

In [ ]:
from google.colab import files

ACTION_LABEL = {
    Action.FORWARD:      'WALK FORWARD',
    Action.LEFT:         'TURN LEFT',
    Action.RIGHT:        'TURN RIGHT',
    Action.STOP:         'STOP — TARGET REACHED',
    Action.SEARCH_LEFT:  'ROTATE LEFT (scanning for target)',
    Action.SEARCH_RIGHT: 'ROTATE RIGHT (scanning for target)',
}
ACTION_COLOR = {
    Action.FORWARD: '#00cc00', Action.LEFT: '#ff8800',
    Action.RIGHT: '#ff8800', Action.STOP: '#cc0000',
    Action.SEARCH_LEFT: '#cc00cc', Action.SEARCH_RIGHT: '#cc00cc',
}
ACTION_ARROW = {
    Action.FORWARD: '\u2191', Action.LEFT: '\u2190',
    Action.RIGHT: '\u2192', Action.STOP: '\u25A0',
    Action.SEARCH_LEFT: '\u21BA', Action.SEARCH_RIGHT: '\u21BB',
}

def run_on_frame(frame):
    """Decide + display for an already-loaded BGR frame. Reusable."""
    t0 = time.monotonic()
    detections = finder.detect(frame)
    action = controller.step(frame)
    elapsed_ms = (time.monotonic() - t0) * 1000
    left, right = pwm_for(action)
    history.append((action, len(detections), elapsed_ms))

    fig = plt.figure(figsize=(14, 11))
    gs = fig.add_gridspec(2, 1, height_ratios=[4, 1.2], hspace=0.05)

    ax_img = fig.add_subplot(gs[0])
    ax_img.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    if detections:
        d = detections[0]
        x1, y1, x2, y2 = d.xyxy
        ax_img.add_patch(Rectangle((x1, y1), x2-x1, y2-y1,
                                   fill=False, edgecolor='lime', linewidth=4))
        ax_img.text(x1, max(y1-8, 16), f'target  {d.confidence:.2f}',
                    color='black', fontsize=14, weight='bold',
                    bbox=dict(facecolor='lime', alpha=0.85, pad=4))
    ax_img.axis('off')

    ax_act = fig.add_subplot(gs[1])
    ax_act.set_xlim(0, 1); ax_act.set_ylim(0, 1); ax_act.axis('off')
    color = ACTION_COLOR[action]
    ax_act.text(0.5, 0.65, f'{ACTION_ARROW[action]}  {ACTION_LABEL[action]}  {ACTION_ARROW[action]}',
                ha='center', va='center', fontsize=44, color=color, weight='bold')
    ax_act.text(0.5, 0.2,
                f'step #{len(history)}   ·   {len(detections)} detection(s)   ·   '
                f'pwm = ({left:+d}, {right:+d})   ·   {elapsed_ms:.0f} ms',
                ha='center', va='center', fontsize=13, color='#666')
    plt.show()
    return action

def take_step():
    print('upload a frame (the robot view — bottle in/out of frame, any pose):')
    uploaded = files.upload()
    if not uploaded:
        print('no file uploaded')
        return None
    name = next(iter(uploaded.keys()))
    frame = cv2.imread(name)
    assert frame is not None, f'could not read {name}'
    return run_on_frame(frame)

_ = take_step()

**Re-run the cell above** to upload another frame. Or batch-test multiple uploaded files at once with the cell below.

In [ ]:
# Batch test — upload several files at once, run the brain on each.
print('select MULTIPLE images (Ctrl/Cmd+click) to batch-test:')
uploaded = files.upload()
for name in uploaded:
    print(f'\n=== {name} ===')
    frame = cv2.imread(name)
    if frame is None:
        print(f'  could not read')
        continue
    run_on_frame(frame)

## Trajectory log

Every action you've gotten this session, in order. Useful for spotting patterns.

In [ ]:
if not history:
    print('no steps taken yet')
else:
    print(f'{"#":>3}  {"action":<14}  {"dets":>4}  {"step ms":>8}')
    print('-' * 38)
    for i, (act, n, ms) in enumerate(history, 1):
        print(f'{i:>3}  {act.name:<14}  {n:>4}  {ms:>8.0f}')
    last5 = [a.name for a, _, _ in history[-5:]]
    if len(last5) == 5 and len(set(last5)) > 3:
        print('\n⚠️  last 5 actions are noisy — controller may be oscillating.')
    if last5 == ['STOP'] * 5:
        print('\n✅ stable STOP — would now begin INTAKING (drive forward into bottle).')

## Reset

In [ ]:
history.clear()
controller._search_remaining = 0
controller._search_direction = None
print('history cleared, controller reset.')

## What this proves

If you upload various frames here and the brain returns sensible Actions (FORWARD when bottle is centered, STOP when bottle fills the frame, SEARCH_* when bottle is hidden, etc.), the **brain code is correct**. Any failure on your local 4080 is then a CUDA / bitsandbytes / driver issue, not our code.

If you upload frames here and the brain returns garbage (everything FORWARD, or everything SEARCH, or random Actions that don't match the visual content), the **wrapper code has a bug**. Tell me what you saw and I'll fix it.